# AI-READI Wasserstein Regression With Qualified Likelihood Diagnostics

This notebook recreates the current AI-READI analytic cohort used in the repository,
focuses the main performance table on DE K=4 versus Wasserstein regression,
and keeps AIC / Clarke diagnostics in view only as conditional plug-in Gaussian summaries.


In [1]:
import importlib.util
import subprocess
import sys

from tools.r_tools import setup_r_environment

setup_r_environment()

if importlib.util.find_spec("rpy2") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "rpy2"])


Using R installation at: C:\Program Files\R\R-4.4.1


In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from statsmodels.formula.api import ols

from tools.r_tools import ensure_r_packages
from tools.ai_readi_tools import format_response_name, load_ai_readi_cohort
from tools.prediction_experim_functions import (
    build_wr_modeling_data,
    clarke_gaussian,
    compute_quantile_matrix,
    fit_wr_scalar_model,
    r_squared,
    pack_lm_model,
)

REPO_ROOT = Path.cwd()
ensure_r_packages(
    ["fdapace", "clarkeTest", "remotes", "WR"],
    github_packages={"WR": "yqgchen/WR"},
)


## Recreate the current AI-READI analytic cohort

The cohort logic below intentionally mirrors the existing `Real-data-ai-readi.ipynb`
workflow so that the Wasserstein comparison stays on the same subjects and outcomes.


In [3]:
filtered_data = load_ai_readi_cohort(REPO_ROOT)

print("Number of subjects:", filtered_data.shape[0])
print("HbA1c range:", filtered_data["hba1c"].min(), "--", filtered_data["hba1c"].max())
print("Subjects with HbA1c < 5.7:", int((filtered_data["hba1c"] < 5.7).sum()))

filtered_data[["id", "hdl_c", "log_triglycerides", "AIP", "hba1c"]].head()


Number of subjects: 573
HbA1c range: 4.5 -- 7.4
Subjects with HbA1c < 5.7: 313


,id,hdl_c,log_triglycerides,AIP,hba1c
0,1001,92.0,4.418841,-0.102948,5.7
1,1002,41.0,5.030438,1.316866,5.6
2,1006,91.0,4.276666,-0.234193,5.4
3,1008,62.0,4.488636,0.361502,6.0
4,1011,41.0,5.924256,2.210684,6.7


In [4]:
responses = ["hdl_c", "log_triglycerides", "AIP"]
probs = np.linspace(0.0, 1.0, 101)

modeling_data, feature_map = build_wr_modeling_data(filtered_data)
qf_matrix = compute_quantile_matrix(filtered_data["gl"], probs)

de_k2 = feature_map["de_k2"]
de_k4 = feature_map["de_k4"]

print("Modeling data shape:", modeling_data.shape)
print("Quantile matrix shape (n x q):", qf_matrix.shape)
print("WR predictor orientation passed to R will be q x n:", qf_matrix.T.shape)


Modeling data shape: (573, 19)
Quantile matrix shape (n x q): (573, 101)
WR predictor orientation passed to R will be q x n: (101, 573)


## Interpretation of AIC and Clarke for Wasserstein Regression

The WR package's scalar-response fit is operationally a two-stage procedure: it first estimates a low-dimensional Wasserstein representation of the distributional predictor and then regresses the centered scalar response on the estimated scores. Because the predictor-side representation and the truncation level are learned from the same data, ordinary AIC and Clarke statistics do not apply to the full Wasserstein-regression pipeline in the same clean way they apply to a single Gaussian `lm` fit.

In this notebook, the reported $\Delta$AIC$_{\mathrm{cond}}$ and Clarke$_{\mathrm{cond}}$ values for Wasserstein regression are therefore conditional plug-in Gaussian diagnostics based on the reconstructed second-stage score regression. They are useful for rough same-sample comparison, but they should be interpreted more cautiously than $R^2$.


In [5]:
responses = ["hdl_c", "log_triglycerides", "AIP"]

lm_results_wr = pd.DataFrame()
comparison_numeric_rows = []
wr_diagnostics_rows = []
de_models = {}
wr_models = {}
clarke_raw = {}

for response in responses:
    formula_de_k4 = f"{response} ~ " + " + ".join(de_k4[:-1])
    model_de_k4 = ols(formula_de_k4, data=modeling_data).fit()
    packed_de_k4 = pack_lm_model(model_de_k4)

    y = modeling_data[response].to_numpy(dtype=float)
    wr_fit = fit_wr_scalar_model(y, qf_matrix, probs)
    wr_cond = wr_fit["conditional_model"].copy()
    wr_cond["r2"] = r_squared(y, wr_fit["fitted"])

    clarke_k4 = clarke_gaussian(packed_de_k4, wr_cond, y)
    label = format_response_name(response)
    delta_aic = packed_de_k4["aic"] - wr_cond["aic"]

    lm_results_wr.at[label, "R^2 (DE K=4)"] = f"{packed_de_k4['r2']:.3f}"
    lm_results_wr.at[label, "R^2 (WR)"] = f"{wr_cond['r2']:.3f}"
    lm_results_wr.at[label, "AIC_cond (DE K=4 - WR, p-value)"] = f"{delta_aic:.1f} ({clarke_k4['p_value']:.3f})"

    comparison_numeric_rows.append(
        {
            "Response": label,
            "R^2 (DE K=4)": packed_de_k4["r2"],
            "R^2 (WR)": wr_cond["r2"],
            "AIC_cond DE (K=4)": packed_de_k4["aic"],
            "AIC_cond WR": wr_cond["aic"],
            "AIC_cond (DE K=4 - WR)": delta_aic,
            "Clarke stat": clarke_k4["stat"],
            "p-value": clarke_k4["p_value"],
        }
    )

    wr_diagnostics_rows.append(
        {
            "Response": label,
            "AIC_cond DE (K=4)": packed_de_k4["aic"],
            "AIC_cond WR": wr_cond["aic"],
            "AIC_cond (DE K=4 - WR)": delta_aic,
            "Clarke stat": clarke_k4["stat"],
            "p-value": clarke_k4["p_value"],
            "WR fit reconstruction max diff": wr_fit["fit_diff_max"],
            "WR score dimension": wr_cond["npar"],
        }
    )

    de_models[label] = model_de_k4
    wr_models[label] = wr_fit
    clarke_raw[label] = clarke_k4

comparison_df = lm_results_wr.copy()
comparison_numeric_df = pd.DataFrame(comparison_numeric_rows).set_index("Response")
wr_diagnostics_df = pd.DataFrame(wr_diagnostics_rows).set_index("Response")

print("Linear model comparison between DE K=4 and Wasserstein regression")
comparison_df


Linear model comparison between DE K=4 and Wasserstein regression


,R^2 (DE K=4),R^2 (WR),"AIC_cond (DE K=4 - WR, p-value)"
HDL-C,0.041,0.041,4.0 (0.000)
TG,0.052,0.054,5.1 (0.000)
TG/HDL-C,0.060,0.067,8.6 (0.000)


In [ ]:
display(Markdown("### Supplementary WR diagnostics"))

diagnostics_display = wr_diagnostics_df.copy()
for column in ["AIC_cond DE (K=4)", "AIC_cond WR", "AIC_cond (DE K=4 - WR)", "p-value", "WR fit reconstruction max diff"]:
    diagnostics_display[column] = diagnostics_display[column].map(lambda value: f"{value:.3f}")
diagnostics_display["Clarke stat"] = diagnostics_display["Clarke stat"].map(lambda value: f"{int(value)}")
diagnostics_display["WR score dimension"] = diagnostics_display["WR score dimension"].map(lambda value: f"{int(value)}")
display(diagnostics_display)

lines = [
    "### Final Interpretation",
    "",
    "The statements below summarize whether Wasserstein regression improves on the settled DE K=4 baseline in $R^2$, conditional $\Delta$AIC, and Clarke's test.",
    "",
]

for response in comparison_numeric_df.index:
    row = comparison_numeric_df.loc[response]
    r2_statement = "improves on" if row["R^2 (WR)"] > row["R^2 (DE K=4)"] else "does not improve on"
    daic_statement = "favors WR" if row["AIC_cond (DE K=4 - WR)"] > 0 else "does not favor WR"
    lines.append(
        f"- **{response}**: WR {r2_statement} DE K=4 in $R^2$; conditional $\Delta$AIC {daic_statement}; Clarke stat = {int(row['Clarke stat'])}, p-value = {row['p-value']:.3f}."
    )

display(Markdown("\n".join(lines)))
